# Cont.IA — Treinamento YOLO11n: Parafuso Philips vs Fenda

> **Antes de começar:** Em `Ambiente de execução → Alterar tipo de ambiente de execução`, selecione **GPU T4**.

---

### Estrutura esperada no Google Drive
```
MyDrive/
└── contia_dataset/
    ├── data.yaml
    ├── images/
    │   ├── train/   (~70% das fotos)
    │   ├── val/     (~20% das fotos)
    │   └── test/    (~10% das fotos)
    └── labels/
        ├── train/   (arquivos .txt gerados pelo Roboflow)
        ├── val/
        └── test/
```

## Passo 1 — Instalar dependências

In [ ]:
!pip install ultralytics roboflow --quiet

import ultralytics
ultralytics.checks()

## Passo 2 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATASET_PATH = '/content/drive/MyDrive/contia_dataset'
DATA_YAML    = f'{DATASET_PATH}/data.yaml'

assert os.path.exists(DATA_YAML), f'data.yaml não encontrado em {DATA_YAML}'
print('Dataset encontrado!')

## Passo 2b — Alternativa: importar direto do Roboflow
> Use esta célula **OU** o Passo 2 acima. Não os dois.

In [ ]:
# from roboflow import Roboflow
#
# rf = Roboflow(api_key='SUA_API_KEY_AQUI')
# project = rf.workspace('seu-workspace').project('parafusos-contia')
# dataset = project.version(1).download('yolov11')
#
# DATA_YAML = dataset.location + '/data.yaml'
# print('Dataset baixado em:', dataset.location)

## Passo 3 — Verificar o dataset

In [ ]:
import glob

splits = ['train', 'val', 'test']
for split in splits:
    imgs = glob.glob(f'{DATASET_PATH}/images/{split}/*.jpg') + \
           glob.glob(f'{DATASET_PATH}/images/{split}/*.png')
    lbls = glob.glob(f'{DATASET_PATH}/labels/{split}/*.txt')
    print(f'{split:5s} → {len(imgs):4d} imagens | {len(lbls):4d} labels')

print()
print('Classes esperadas: parafuso_philips (0), parafuso_fenda (1)')

## Passo 4 — Treinamento YOLO11n

| Parâmetro | Valor | Motivo |
|-----------|-------|--------|
| `model` | yolo11n.pt | Nano — leve, rápido, ideal para mobile |
| `epochs` | 100 | Suficiente para 2 classes |
| `imgsz` | 640 | Padrão YOLO |
| `batch` | 16 | Ajuste automático se der OOM |
| `patience` | 20 | Para cedo se não melhorar |

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    name='contia_parafusos',
    project='/content/drive/MyDrive/contia_runs',
    exist_ok=True,
    device=0,           # GPU
    workers=4,
    verbose=True,
)

print('Treinamento concluído!')
print('Melhor modelo salvo em:', results.save_dir)

## Passo 5 — Avaliar o modelo

In [ ]:
best_model_path = f'/content/drive/MyDrive/contia_runs/contia_parafusos/weights/best.pt'

model_best = YOLO(best_model_path)
metrics = model_best.val(data=DATA_YAML)

print(f'mAP50      : {metrics.box.map50:.3f}')
print(f'mAP50-95   : {metrics.box.map:.3f}')
print(f'Precisão   : {metrics.box.mp:.3f}')
print(f'Recall     : {metrics.box.mr:.3f}')

## Passo 6 — Testar com uma imagem

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

# Pega uma imagem de teste aleatória
test_images = glob.glob(f'{DATASET_PATH}/images/test/*.jpg') + \
              glob.glob(f'{DATASET_PATH}/images/test/*.png')

if test_images:
    img_path = test_images[0]
    resultado = model_best.predict(img_path, conf=0.25, save=False)[0]

    print(f'Imagem: {img_path}')
    print(f'Total detectado: {len(resultado.boxes)}')

    for box in resultado.boxes:
        cls  = int(box.cls.item())
        conf = box.conf.item()
        nome = resultado.names[cls]
        print(f'  → {nome} (confiança: {conf:.2f})')

    # Exibir imagem com bounding boxes
    plotted = resultado.plot()
    plt.figure(figsize=(10, 8))
    plt.imshow(plotted[..., ::-1])
    plt.axis('off')
    plt.title('Detecções — Cont.IA')
    plt.show()
else:
    print('Nenhuma imagem de teste encontrada.')

## Passo 7 — Exportar para o backend

O arquivo `best.pt` gerado é o modelo final.

**Para usar no Cont.IA:**
1. Baixe o arquivo `best.pt` do Google Drive
2. Renomeie para `contia_parafusos.pt`
3. Coloque na pasta `backend/`
4. Atualize o `backend/.env`:
```
YOLO_MODEL=contia_parafusos.pt
```
5. Reinicie o backend: `docker compose up --build`

In [ ]:
print('Caminho do melhor modelo:')
print(best_model_path)
print()
print('Para baixar, acesse o Google Drive em:')
print('  MyDrive/contia_runs/contia_parafusos/weights/best.pt')

---
## Referência: como interpretar as métricas

| mAP50 | Interpretação |
|-------|---------------|
| < 0.50 | Insuficiente — coletar mais dados |
| 0.50 – 0.70 | Razoável — melhorar dataset |
| 0.70 – 0.85 | Bom — pronto para testes reais |
| > 0.85 | Ótimo — pronto para produção |

**Dicas se o mAP estiver baixo:**
- Adicionar mais imagens (especialmente do tipo que errou mais)
- Variar ângulo, iluminação e fundo nas fotos
- Aumentar `epochs` para 150
- Usar `yolo11s.pt` (Small) em vez de Nano se tiver GPU melhor